# Pancancer enrichment analysis step 1: Find genes with significant change

In [1]:
import cptac
import cptac.utils as ut
import statsmodels
import numpy as np
import pandas as pd

In [2]:
# Create a dictionary of the datasets
# Don't load them yet--we'll do it one at a time to save RAM
dss = {
#     "brca": cptac.Brca, # Exclude BRCA--no normal samples
    "ccrcc": cptac.Ccrcc,
    "colon": cptac.Colon,
    "endometrial": cptac.Endometrial,
    "gbm": cptac.Gbm,
    "hnscc": cptac.Hnscc,
    "lscc": cptac.Lscc,
    "luad": cptac.Luad,
    "ovarian": cptac.Ovarian,
}

In [3]:
# We define this as a separate function so that its objects will
# go out of scope each time the function returns, which will
# clear up RAM

def test_tumor_changes(dataset):
    ds = dataset(no_internet=True) # No internet b/c supercomputer has no internet connection
    
    # Get the proteomics data
    prot_tumor = ds.get_proteomics(tissue_type="tumor")
    prot_normal = ds.get_proteomics(tissue_type="normal")
    
    # Run the permutation test for each gene
    proteins = []
    changes = []
    p_vals = []
    
    for protein in prot_tumor.columns:
        
        # Get the data for the protein
        sample_tumor = prot_tumor[protein]
        sample_normal = prot_normal[protein]
        
        # Skip it if either array is all null
        if sample_tumor.isna().all() or sample_normal.isna().all():
            continue
        
        change, p_val = ut.permutation_test_means(
            sample_tumor,
            sample_normal,
            num_permutations=10)
        
        proteins.append(protein)
        changes.append(change)
        p_vals.append(p_val)
        
    results = pd.DataFrame({
        "protein": proteins,
        "change": changes, 
        "P_val": p_vals})
    
    return results

In [4]:
tumor_changes = pd.DataFrame()

for cancer_type, dataset in dss.items():
    cancer_type_changes = test_tumor_changes(dataset)
    cancer_type_changes.insert(0, "cancer_type", cancer_type)
    tumor_changes = tumor_changes.append(cancer_type_changes)
    
tumor_changes = tumor_changes.reset_index(drop=True)

cptac warning: The GBM dataset is under publication embargo until March 01, 2021. CPTAC is a community resource project and data are made available rapidly after generation for community research use. The embargo allows exploring and utilizing the data, but analysis may not be published until after the embargo date. Please see https://proteomics.cancer.gov/data-portal/about/data-use-agreement or enter cptac.embargo() to open the webpage for more details. (/home/caleb/anaconda3/envs/dev/lib/python3.7/site-packages/ipykernel_launcher.py, line 6)


cptac warning: The HNSCC data is currently strictly reserved for CPTAC investigators. Otherwise, you are not authorized to access these data. Additionally, even after these data become publicly available, they will be subject to a publication embargo (see https://proteomics.cancer.gov/data-portal/about/data-use-agreement or enter cptac.embargo() to open the webpage for more details). (/home/caleb/anaconda3/envs/dev/lib/python3.7/site-packages/ipykernel_launcher.py, line 6)


cptac warning: The LSCC data is currently strictly reserved for CPTAC investigators. Otherwise, you are not authorized to access these data. Additionally, even after these data become publicly available, they will be subject to a publication embargo (see https://proteomics.cancer.gov/data-portal/about/data-use-agreement or enter cptac.embargo() to open the webpage for more details). (/home/caleb/anaconda3/envs/dev/lib/python3.7/site-packages/ipykernel_launcher.py, line 6)


cptac warning: The LUAD dataset is under publication embargo until July 01, 2020. CPTAC is a community resource project and data are made available rapidly after generation for community research use. The embargo allows exploring and utilizing the data, but analysis may not be published until after the embargo date. Please see https://proteomics.cancer.gov/data-portal/about/data-use-agreement or enter cptac.embargo() to open the webpage for more details. (/home/caleb/anaconda3/envs/dev/lib/python3.7/site-packages/ipykernel_launcher.py, line 6)


In [5]:
# Multiple testing correction
reject_null, adj_p, alpha_sidak, alpha_bonf = statsmodels.stats.multitest.multipletests(
    pvals=tumor_changes["P_val"],
    alpha=0.05,
    method="fdr_bh")

tumor_changes = tumor_changes.assign(adj_p=adj_p, reject_null=reject_null)

# Separate out proteins where we rejected the null hypothesis
rej_null = tumor_changes[tumor_changes.reject_null]

# For rows with tuples as the protein names, split out just the protein names
protein_str = rej_null["protein"].apply(lambda x: x[0] if type(x) == tuple else x)
rej_null = rej_null.assign(protein_str=protein_str)

In [8]:
# Save the results
rej_null.to_csv("reject_null.tsv", sep='\t')